In [ ]:
!pip install playwright nest-asyncio tenacity
!playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 MB 10.8 MB/s eta 0:00:00
164.9 MiB [] 0% 0.0s164.9 MiB [] 0% 50.6s164.9 MiB [] 0% 29.0s164.9 MiB [] 0% 17.9s164.9 MiB [] 0% 14.9s164.9 MiB [] 0% 9.8s164.9 MiB [] 1% 7.4s164.9 MiB [] 1% 6.0s164.9 MiB [] 2% 4.9s164.9 MiB [] 3% 4.6s164.9 MiB [] 3% 4.1s164.9 MiB [] 4% 3.7s164.9 MiB [] 5% 3.5s164.9 MiB [] 6% 3.3s164.9 MiB [] 6% 3.4s164.9 MiB [] 6% 3.5s164.9 MiB [] 7% 3.5s164.9 MiB [] 7% 3.3s164.9 MiB [] 8% 3.4s164.9 MiB [] 8% 3.3s164.9 MiB [] 9% 3.1s164.9 MiB [] 10% 3.1s164.9 MiB [] 11% 3.1s164.9 MiB [] 11% 3.2s164.9 MiB [] 11% 3.1s164.9 MiB [] 12% 3.1s164.9 MiB [] 13% 3.0s164.9 MiB [] 14% 2.9s164.9 MiB [] 15% 2.8s164.9 MiB [] 16% 2.8s164.9 MiB [] 16% 2.7s164.9 MiB [] 17% 2.7s164.9 MiB [] 18% 2.6s164.9 MiB [] 19% 2.5s164.9 MiB [] 20% 2.5s164.9 MiB [] 21% 2.4s164.9 MiB [] 22% 2.3s164.9 MiB [] 22% 2.4s164.9 MiB [] 24% 2.3s164.9 MiB [] 25% 2.2s164.9 MiB [] 26% 2.1s164.9 MiB [] 27% 2.0s164.9 MiB [] 28% 2.0s164.9 MiB [] 29% 1.9s164.9 MiB [] 3

In [ ]:
import asyncio
import json
from playwright.async_api import async_playwright

regions = {
    "south indian": "South Indian",
    "north indian": "North Indian",
    "bengali": "Bengal",
    "gujarati": "Gujarati",
    "rajasthani": "Rajasthani",
    "maharashtrian": "Maharashtrian",
    "punjabi": "Punjabi"
}

async def scrape_region_recipes(region_query, region_name, page_limit=3):
    recipe_data = []
    failed_urls = []

    async with async_playwright() as p:
        # Launch browser with slower timeout settings
        browser = await p.chromium.launch(
            headless=True,
            timeout=60000
        )
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            viewport={'width': 1280, 'height': 720}
        )

        # Slow down the browser to appear more human-like
        context.set_default_timeout(45000)
        context.set_default_navigation_timeout(60000)

        page = await context.new_page()

        for page_num in range(1, page_limit + 1):
            search_url = f"https://www.tarladalal.com/recipesearch/?query={region_query}&page={page_num}"
            try:
                print(f"Scraping page {page_num} for {region_name}")
                await page.goto(search_url, wait_until="domcontentloaded", timeout=60000)

                # Wait for recipe links to load
                await page.wait_for_selector("h5.mb-0.two-line-text a", timeout=30000)

                recipe_links = await page.query_selector_all("h5.mb-0.two-line-text a")
                recipe_urls = ["https://www.tarladalal.com" + await link.get_attribute("href") for link in recipe_links]

                print(f"Found {len(recipe_urls)} recipes on page {page_num}")

                for url in recipe_urls:
                    try:
                        print(f"Scraping recipe: {url}")
                        await page.goto(url, wait_until="domcontentloaded", timeout=60000)

                        # Wait for essential elements to load
                        await page.wait_for_selector("h1", timeout=30000)
                        await page.wait_for_selector("div#ingredients", timeout=30000)
                        await page.wait_for_selector("div.methods", timeout=30000)

                        dish_name = await page.locator("h1").text_content()
                        dish_name = dish_name.strip() if dish_name else "Unknown"

                        ingredients_section = await page.query_selector("div#ingredients")
                        method_section = await page.query_selector("div.methods")

                        # Extract ingredients
                        ingredients_raw = await ingredients_section.inner_text() if ingredients_section else ""
                        ingredients = [line.strip() for line in ingredients_raw.split('\n') if line.strip()]

                        # Extract instructions
                        instructions_raw = await method_section.inner_text() if method_section else ""
                        instructions = [line.strip() for line in instructions_raw.split('\n') if line.strip()]

                        recipe_data.append({
                            "region": region_name,
                            "name": dish_name,
                            "ingredients": ingredients,
                            "instructions": instructions,
                            "url": url
                        })

                    except Exception as e:
                        print(f"Error scraping recipe {url}: {str(e)}")
                        failed_urls.append(url)
                        continue

            except Exception as e:
                print(f"Error scraping page {page_num} for {region_name}: {str(e)}")
                continue

        await browser.close()

    if failed_urls:
        print(f"Failed to scrape {len(failed_urls)} URLs for {region_name}")
        with open(f"failed_urls_{region_name}.txt", "w") as f:
            f.write("\n".join(failed_urls))

    return recipe_data


async def main():
    all_data = []
    failed_regions = []

    for query, region in regions.items():
        try:
            print(f"\nStarting to scrape for region: {region}")
            region_data = await scrape_region_recipes(query, region, page_limit=2)
            all_data.extend(region_data)
            print(f"Completed scraping for {region}. Found {len(region_data)} recipes.")
        except Exception as e:
            print(f"Major error scraping region {region}: {str(e)}")
            failed_regions.append(region)
            continue

    # Save to JSON
    with open("tarla_dalal_recipes.json", "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    if failed_regions:
        print(f"\nFailed to scrape these regions: {', '.join(failed_regions)}")

    print("\n✅ Scraping complete. Saved as 'tarla_dalal_recipes.json'")
    print(f"Total recipes collected: {len(all_data)}")

if __name__ == "__main__":
    asyncio.run(main())

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed



Starting to scrape for region: South Indian
Scraping page 1 for South Indian
Found 60 recipes on page 1
Scraping recipe: https://www.tarladalal.com/coconut-chutney---idlis-and-dosas-1653r
Scraping recipe: https://www.tarladalal.com/sambar-recipe-south-indian-homemade-sambar-recipe-1557r
Scraping recipe: https://www.tarladalal.com/dal-makhani-30900r
Scraping recipe: https://www.tarladalal.com/medu-vada--south-indian-recipe-32683r
Scraping recipe: https://www.tarladalal.com/rice-appe--how-to-make-rice-appe--32847r
Scraping recipe: https://www.tarladalal.com/curd-rice-south-indian-curd-rice-recipe-32893r
Scraping recipe: https://www.tarladalal.com/dosa---south-indian-recipe-32927r
Scraping recipe: https://www.tarladalal.com/sambar--sambhar-idlis-and-dosas-1663r
Scraping recipe: https://www.tarladalal.com/how-to-make-a-perfect-dosa-batter-40481r
Scraping recipe: https://www.tarladalal.com/idli-1652r
Scraping recipe: https://www.tarladalal.com/rava-dosa-onion-rava-dosa-169r
Scraping recipe

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed
ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


Scraping recipe: https://www.tarladalal.com/upma--quick-upma-recipe-breakfast-upma-38658r
Scraping recipe: https://www.tarladalal.com/sooji-idli--suji-idli-1384r
Scraping recipe: https://www.tarladalal.com/idli--how-to-make-idli--32833r
Scraping recipe: https://www.tarladalal.com/quick-rava-idli--south-indian-recipes-32608r
Scraping recipe: https://www.tarladalal.com/broken-wheat-upma-healthy-dalia-upma-4650r
Scraping recipe: https://www.tarladalal.com/appam-appam-kerala-recipe-193r
Scraping recipe: https://www.tarladalal.com/tomato-rice--south-indian-recipes--32889r
Scraping recipe: https://www.tarladalal.com/coconut-and-rava-ladoo---laddu-36287r
Scraping recipe: https://www.tarladalal.com/rajma-curry--punjabi-rajma-masala-recipe-1539r
Scraping recipe: https://www.tarladalal.com/rava-idli-in-microwave-4883r
Scraping recipe: https://www.tarladalal.com/gatte-ki-kadhi-recipe-4344r
Scraping recipe: https://www.tarladalal.com/rava-dosa-how-to-make-rava-dosa-32837r
Scraping recipe: https://

In [ ]:
import json

# Load the scraped file
with open("tarla_dalal_recipes.json", "r", encoding="utf-8") as f:
    recipes = json.load(f)

# Fix unknown names
for dish in recipes:
    if dish["name"] == "Unknown":
        if dish.get("instructions") and dish["instructions"]:
            dish["name"] = dish["instructions"][0].strip()
        elif dish.get("ingredients") and dish["ingredients"]:
            dish["name"] = dish["ingredients"][0].strip()

# Save the fixed file
with open("tarla_dalal_recipes_fixed.json", "w", encoding="utf-8") as f:
    json.dump(recipes, f, ensure_ascii=False, indent=2)

print("✅ Fixed dataset saved as 'tarla_dalal_recipes_fixed.json'")


✅ Fixed dataset saved as 'tarla_dalal_recipes_fixed.json'


In [ ]:
import json
from collections import Counter

# Load the fixed dataset
with open("tarla_dalal_recipes_fixed.json", "r", encoding="utf-8") as f:
    recipes = json.load(f)

# Count recipes per region
region_counts = Counter(dish["region"] for dish in recipes)

# Print nicely
for region, count in region_counts.items():
    print(f"{region}: {count} recipes")


South Indian: 119 recipes
North Indian: 120 recipes
Bengal: 120 recipes
Gujarati: 120 recipes
Rajasthani: 119 recipes
Maharashtrian: 118 recipes
Punjabi: 120 recipes
